In [1]:
import numpy as np
import xcdat as xc
import scipy
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt 
from cdo import *   # python version
import scipy.stats as stats
import os

sys.path.append('../../functions/')
from lag_linregress import *
from monthly_departures import *
from MCA import *
cdo = Cdo()

import shutil
import tempfile
from pathlib import Path

In [2]:
# Cell 1 — Configuration, ETH inventory, and source validation
# ETH historical processing configuration.
#
# The ETH monthly files are already on the desired g025 grid. No remapping
# operator is used anywhere in this HIST pipeline.

from pathlib import Path
from collections import defaultdict
import re
import shutil
import subprocess
import tempfile

import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date


eth_manifest_path = Path(
    "./remote_structure_manifest_maria_ETH.txt"
)

# ETH's original Next_Generation.v2 layout is:
#   <root>/<variable>/mon/g025/<filename>
#
# If the files instead live in transferred directories such as
# tas_latlon_mon/, set this to their common parent. The resolver below
# supports either layout.
eth_source_root = Path("/rugenstein-archive/mariarug/model_output/cmip6_fromETH/")

# HIST will sit alongside AMIP, CERES, and ERA5.
hist_output_root = Path("./data/HIST")

# Reference used only to confirm that ETH's existing grid equals the grid
# used by the final AMIP products. It is never used for remapping here.
hist_g025_reference = Path(
    "/scratch/leiff/MCA/data/"
    "rlut_mon_CanESM5_historical_r1i1p1f1_g025.nc"
)

hist_start_date = "1870-01-01"
hist_end_date = "2014-12-31"
hist_output_period = "187001-201412"
hist_overwrite = False

required_eth_variables = {
    "tas",
    "rsdt",
    "rlut",
    "rsut",
}


def hist_member_sort_key(member):
    """Numerically sort all r, i, p, and f components."""
    match = re.fullmatch(
        r"r(\d+)i(\d+)p(\d+)(?:f(\d+))?",
        str(member),
    )

    if match is None:
        return (float("inf"), str(member))

    realization, initialization, physics, forcing = match.groups()

    return (
        int(realization),
        int(initialization),
        int(physics),
        int(forcing) if forcing is not None else 0,
    )


def resolve_eth_source(relative_path, variable):
    """
    Resolve and validate an ETH source file.

    Empty files, unreadable NetCDF files, and files lacking the expected
    variable are treated as unavailable.
    """
    relative_path = Path(relative_path)
    filename = relative_path.name

    candidates = [
        eth_source_root / relative_path,
        eth_source_root / variable / "mon" / "g025" / filename,
        eth_source_root / filename,
    ]

    unique_candidates = list(dict.fromkeys(candidates))
    invalid_candidates = []

    for candidate in unique_candidates:
        if not candidate.is_file():
            continue

        if candidate.stat().st_size == 0:
            invalid_candidates.append(
                f"{candidate}: empty file"
            )
            continue

        try:
            # Opening only the header is enough to detect most corrupt files.
            with Dataset(candidate, "r") as dataset:
                if variable not in dataset.variables:
                    invalid_candidates.append(
                        f"{candidate}: missing variable {variable}"
                    )
                    continue

        except OSError as exc:
            invalid_candidates.append(
                f"{candidate}: unreadable NetCDF ({exc})"
            )
            continue

        return candidate

    details = (
        "\n".join(f"  {message}" for message in invalid_candidates)
        if invalid_candidates
        else "\n".join(f"  not found: {path}" for path in unique_candidates)
    )

    raise OSError(
        f"No usable ETH source for {filename}:\n{details}"
    )


eth_filename_pattern = re.compile(
    r"^\./"
    r"(?P<variable>tas|rsdt|rlut|rsut)_latlon_mon/"
    r"(?P=variable)_mon_"
    r"(?P<model>.+)_historical_"
    r"(?P<member>r\d+i\d+p\d+(?:f\d+)?)_"
    r"(?P<grid>g025)\.nc$"
)

eth_records = []

with eth_manifest_path.open() as manifest:
    for line in manifest:
        manifest_entry = line.strip()
        match = eth_filename_pattern.match(manifest_entry)

        if match is None:
            continue

        record = match.groupdict()
        record["relative_path"] = manifest_entry.removeprefix("./")
        eth_records.append(record)

eth_files = pd.DataFrame(eth_records).drop_duplicates()

if eth_files.empty:
    raise RuntimeError(
        f"No monthly historical g025 files found in "
        f"{eth_manifest_path}"
    )

# A model/member/variable must resolve to exactly one source file.
duplicate_sources = (
    eth_files
    .groupby(["model", "member", "variable"])
    .size()
)

duplicate_sources = duplicate_sources.loc[
    duplicate_sources > 1
]

if not duplicate_sources.empty:
    raise RuntimeError(
        "The ETH manifest contains duplicate model/member/variable "
        f"records:\n{duplicate_sources}"
    )

eth_hist_tasks_by_model = defaultdict(list)
incomplete_eth_members = []
unresolved_eth_sources = []

for (model, member), group in eth_files.groupby(
    ["model", "member"]
):
    variables = set(group["variable"])

    if not required_eth_variables.issubset(variables):
        incomplete_eth_members.append(
            {
                "model": model,
                "member": member,
                "missing": sorted(
                    required_eth_variables - variables
                ),
            }
        )
        continue

    source_files = {}

    for row in group.itertuples(index=False):
        if row.variable not in required_eth_variables:
            continue

        try:
            source_files[row.variable] = resolve_eth_source(
                row.relative_path,
                row.variable,
            )
        except (FileNotFoundError, OSError) as exc:
            unresolved_eth_sources.append(str(exc))

    if set(source_files) == required_eth_variables:
        eth_hist_tasks_by_model[model].append(
            {
                "model": model,
                "experiment": "historical",
                "member": member,
                "sources": source_files,
            }
        )

for model in eth_hist_tasks_by_model:
    eth_hist_tasks_by_model[model].sort(
        key=lambda task: hist_member_sort_key(task["member"])
    )

print(
    "ETH historical members with tas, rsdt, rlut, and rsut:"
)

total_members = 0

for model, tasks in sorted(eth_hist_tasks_by_model.items()):
    total_members += len(tasks)
    print(
        f"  {'historical':<16}"
        f"{model:<28}"
        f"{len(tasks):>3} member(s)"
    )

print(
    f"\nTotal: {len(eth_hist_tasks_by_model)} model(s), "
    f"{total_members} model-member combination(s)"
)

if unresolved_eth_sources:
    print("\nUnavailable ETH source files; affected members were skipped:")

    for message in unresolved_eth_sources:
        print(message)

if not eth_hist_tasks_by_model:
    raise RuntimeError(
        "No complete ETH historical members remain after source validation."
    )

# Confirm that a representative ETH source has the exact same g025
# latitude and longitude coordinates as the AMIP target grid.
representative_task = next(
    iter(next(iter(eth_hist_tasks_by_model.values())))
)
representative_file = representative_task["sources"]["rlut"]

with (
    Dataset(representative_file, "r") as eth_dataset,
    Dataset(hist_g025_reference, "r") as reference_dataset,
):
    for coordinate in ("lat", "lon"):
        if coordinate not in eth_dataset.variables:
            raise KeyError(
                f"{representative_file} has no {coordinate} coordinate"
            )

        if coordinate not in reference_dataset.variables:
            raise KeyError(
                f"{hist_g025_reference} has no {coordinate} coordinate"
            )

        if not np.allclose(
            eth_dataset.variables[coordinate][:],
            reference_dataset.variables[coordinate][:],
            equal_nan=True,
        ):
            raise ValueError(
                f"ETH and AMIP {coordinate} coordinates differ"
            )

print(
    f"\nGrid check passed using {representative_file.name}; "
    "no remapping is needed."
)

ETH historical members with tas, rsdt, rlut, and rsut:
  historical      ACCESS-CM2                    3 member(s)
  historical      ACCESS-ESM1-5                30 member(s)
  historical      AWI-CM-1-1-MR                 5 member(s)
  historical      AWI-ESM-1-1-LR                1 member(s)
  historical      BCC-CSM2-MR                   3 member(s)
  historical      BCC-ESM1                      3 member(s)
  historical      CAMS-CSM1-0                   3 member(s)
  historical      CAS-ESM2-0                    4 member(s)
  historical      CESM2                        11 member(s)
  historical      CESM2-FV2                     3 member(s)
  historical      CESM2-WACCM                   3 member(s)
  historical      CESM2-WACCM-FV2               3 member(s)
  historical      CIESM                         3 member(s)
  historical      CMCC-CM2-HR4                  1 member(s)
  historical      CMCC-CM2-SR5                  1 member(s)
  historical      CMCC-ESM2                  

In [5]:
# Cell 2 — Create member files and combine labeled ensembles
def run_hist_cdo(command, description):
    """Run CDO with useful diagnostics if a command fails."""
    result = subprocess.run(
        command,
        text=True,
        capture_output=True,
    )

    if result.returncode != 0:
        print("\nCDO command:")
        print(" ".join(command))
        print("\nCDO stderr:")
        print(result.stderr)
        raise RuntimeError(description)


def create_eth_member_n(task, output_file):
    """
    Calculate N = rsdt - rlut - rsut and select 1870–2014.

    The ETH inputs are already g025, so this contains no remap operator.
    """
    temporary_directory = Path(
        tempfile.mkdtemp(prefix="eth_N_inputs_")
    )

    try:
        merged_file = temporary_directory / "radiation_merged.nc"

        merge_command = [
            "cdo",
            "-L",
            "-O",
            "merge",
            str(task["sources"]["rsdt"]),
            str(task["sources"]["rlut"]),
            str(task["sources"]["rsut"]),
            str(merged_file),
        ]

        run_hist_cdo(
            merge_command,
            (
                f"CDO merge failed for {task['model']} / "
                f"{task['member']}"
            ),
        )

        calculate_command = [
            "cdo",
            "-L",
            "-O",
            "-f", "nc4c",
            "-z", "zip_4",
            f"-seldate,{hist_start_date},{hist_end_date}",
            "-expr,N=rsdt-rlut-rsut",
            str(merged_file),
            str(output_file),
        ]

        run_hist_cdo(
            calculate_command,
            (
                f"CDO N calculation failed for {task['model']} / "
                f"{task['member']}"
            ),
        )

    finally:
        shutil.rmtree(
            temporary_directory,
            ignore_errors=True,
        )


def create_eth_member_tas(task, output_file):
    """
    Copy spatial tas and select 1870–2014.

    The ETH source is already g025, so this also contains no remap operator.
    """
    command = [
        "cdo",
        "-L",
        "-O",
        "-f", "nc4c",
        "-z", "zip_4",
        f"-seldate,{hist_start_date},{hist_end_date}",
        "-selname,tas",
        str(task["sources"]["tas"]),
        str(output_file),
    ]

    run_hist_cdo(
        command,
        (
            f"CDO tas selection failed for {task['model']} / "
            f"{task['member']}"
        ),
    )


def hist_month_sequence(dataset):
    """Return the monthly time coordinate as comparable YYYYMM integers."""
    time = dataset.variables["time"]
    calendar = getattr(time, "calendar", "standard")

    dates = num2date(
        time[:],
        units=time.units,
        calendar=calendar,
    )

    return tuple(
        date.year * 12 + date.month - 1
        for date in dates
    )


expected_hist_months = tuple(
    range(
        1870 * 12,
        2014 * 12 + 12,
    )
)


def validate_eth_member_file(path, variable):
    """Ensure a temporary member contains the exact requested window."""
    with Dataset(path, "r") as dataset:
        if variable not in dataset.variables:
            raise KeyError(
                f"{path} contains no {variable} variable"
            )

        if hist_month_sequence(dataset) != expected_hist_months:
            raise ValueError(
                f"{path} does not contain exactly "
                "1870-01 through 2014-12"
            )

        variable_dimensions = dataset.variables[variable].dimensions

        if variable_dimensions != ("time", "lat", "lon"):
            raise ValueError(
                f"Unexpected {variable} dimensions in {path}: "
                f"{variable_dimensions}"
            )


def copy_hist_member_to_combined(
    source_file,
    combined_file,
    variable,
    member_index,
    member_name,
    model,
):
    """
    Append one ETH member to a labeled, compressed combined NetCDF.

    The output variable is:
      N(member, time, lat, lon)
    or:
      tas(member, time, lat, lon)
    """
    with Dataset(source_file, "r") as source:
        source_variable = source.variables[variable]
        source_dimensions = source_variable.dimensions

        if not combined_file.exists():
            with Dataset(
                combined_file,
                "w",
                format="NETCDF4",
            ) as destination:
                destination.createDimension("member", None)

                for dimension in source_dimensions:
                    destination.createDimension(
                        dimension,
                        len(source.dimensions[dimension]),
                    )

                # Copy time, lat, and lon coordinate variables.
                for dimension in source_dimensions:
                    source_coordinate = source.variables[dimension]
                    coordinate_options = {}

                    if "_FillValue" in source_coordinate.ncattrs():
                        coordinate_options["fill_value"] = (
                            source_coordinate.getncattr("_FillValue")
                        )

                    destination_coordinate = (
                        destination.createVariable(
                            dimension,
                            source_coordinate.dtype,
                            (dimension,),
                            **coordinate_options,
                        )
                    )

                    destination_coordinate[:] = source_coordinate[:]

                    for attribute in source_coordinate.ncattrs():
                        if attribute != "_FillValue":
                            destination_coordinate.setncattr(
                                attribute,
                                source_coordinate.getncattr(
                                    attribute
                                ),
                            )

                member_coordinate = destination.createVariable(
                    "member",
                    str,
                    ("member",),
                )
                member_coordinate.long_name = (
                    "CMIP6 ensemble member identifier"
                )
                member_coordinate.comment = (
                    "Members are ordered numerically by realization, "
                    "initialization, physics, and forcing identifiers."
                )

                output_variable = destination.createVariable(
                    variable,
                    "f4",
                    ("member",) + source_dimensions,
                    zlib=True,
                    complevel=4,
                    chunksizes=(
                        1,
                        min(12, len(source.dimensions["time"])),
                        len(source.dimensions["lat"]),
                        len(source.dimensions["lon"]),
                    ),
                    fill_value=np.float32(np.nan),
                )

                for attribute in source_variable.ncattrs():
                    if attribute != "_FillValue":
                        output_variable.setncattr(
                            attribute,
                            source_variable.getncattr(attribute),
                        )

                if variable == "N":
                    output_variable.long_name = (
                        "Net top-of-atmosphere radiation"
                    )
                    output_variable.N_definition = (
                        "rsdt - rlut - rsut"
                    )
                else:
                    output_variable.long_name = getattr(
                        source_variable,
                        "long_name",
                        "Near-surface air temperature",
                    )

                output_variable.source_grid = "g025"
                output_variable.regridding = (
                    "None; ETH source data were already on g025"
                )

                destination.model = model
                destination.experiment = "historical"
                destination.source_collection = (
                    "ETH Next_Generation.v2 monthly g025"
                )
                destination.source_grid = "g025"
                destination.regridding = (
                    "None; existing ETH g025 fields copied without "
                    "spatial interpolation"
                )
                destination.time_selection = (
                    "1870-01 through 2014-12"
                )

        with Dataset(combined_file, "a") as destination:
            # Confirm every new member has the same dimensions.
            for dimension in source_dimensions:
                if (
                    len(destination.dimensions[dimension])
                    != len(source.dimensions[dimension])
                ):
                    raise ValueError(
                        f"{model} {member_name} {variable}: "
                        f"{dimension} length differs between members"
                    )

            destination.variables["member"][
                member_index
            ] = member_name

            destination_variable = destination.variables[variable]
            number_of_months = len(source.dimensions["time"])

            # Copy one year at a time to keep memory use small.
            for first_time in range(0, number_of_months, 12):
                last_time = min(
                    first_time + 12,
                    number_of_months,
                )

                destination_variable[
                    member_index,
                    first_time:last_time,
                    :,
                    :,
                ] = source_variable[
                    first_time:last_time,
                    :,
                    :,
                ]

In [3]:
# Cell 3 — Write all combined HIST N files
for model, tasks in sorted(eth_hist_tasks_by_model.items()):
    model_output_directory = hist_output_root / model
    model_output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    final_file = (
        model_output_directory
        / (
            f"N_Amon_{model}_historical_combined_"
            f"{hist_output_period}_g025.nc"
        )
    )
    partial_file = final_file.with_suffix(".partial.nc")

    if final_file.exists() and not hist_overwrite:
        print(f"Skipping existing file: {final_file}")
        continue

    if partial_file.exists():
        partial_file.unlink()

    print(f"\n{model} / historical: {len(tasks)} member(s)")

    try:
        with tempfile.TemporaryDirectory(
            prefix="eth_hist_N_"
        ) as temporary_directory:
            temporary_directory = Path(temporary_directory)

            for member_index, task in enumerate(tasks):
                member = task["member"]
                temporary_member_file = (
                    temporary_directory / f"{member}_N.nc"
                )

                print(f"    N: {member}")

                create_eth_member_n(
                    task,
                    temporary_member_file,
                )
                validate_eth_member_file(
                    temporary_member_file,
                    "N",
                )
                copy_hist_member_to_combined(
                    temporary_member_file,
                    partial_file,
                    "N",
                    member_index,
                    member,
                    model,
                )

                temporary_member_file.unlink()

        partial_file.replace(final_file)

        print(f"  Saved: {final_file}")
        subprocess.run(
            ["du", "-h", str(final_file)],
            check=True,
        )

    except Exception:
        print(
            f"  FAILED: incomplete output retained at "
            f"{partial_file}"
        )
        raise

Skipping existing file: data/HIST/ACCESS-CM2/N_Amon_ACCESS-CM2_historical_combined_187001-201412_g025.nc
Skipping existing file: data/HIST/ACCESS-ESM1-5/N_Amon_ACCESS-ESM1-5_historical_combined_187001-201412_g025.nc
Skipping existing file: data/HIST/AWI-CM-1-1-MR/N_Amon_AWI-CM-1-1-MR_historical_combined_187001-201412_g025.nc
Skipping existing file: data/HIST/AWI-ESM-1-1-LR/N_Amon_AWI-ESM-1-1-LR_historical_combined_187001-201412_g025.nc
Skipping existing file: data/HIST/BCC-CSM2-MR/N_Amon_BCC-CSM2-MR_historical_combined_187001-201412_g025.nc
Skipping existing file: data/HIST/BCC-ESM1/N_Amon_BCC-ESM1_historical_combined_187001-201412_g025.nc
Skipping existing file: data/HIST/CAMS-CSM1-0/N_Amon_CAMS-CSM1-0_historical_combined_187001-201412_g025.nc
Skipping existing file: data/HIST/CAS-ESM2-0/N_Amon_CAS-ESM2-0_historical_combined_187001-201412_g025.nc
Skipping existing file: data/HIST/CESM2/N_Amon_CESM2_historical_combined_187001-201412_g025.nc
Skipping existing file: data/HIST/CESM2-FV2/N

In [6]:
# Cell 4 — Write all combined HIST tas files
for model, tasks in sorted(eth_hist_tasks_by_model.items()):
    model_output_directory = hist_output_root / model
    model_output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    final_file = (
        model_output_directory
        / (
            f"tas_Amon_{model}_historical_combined_"
            f"{hist_output_period}_g025.nc"
        )
    )
    partial_file = final_file.with_suffix(".partial.nc")

    if final_file.exists() and not hist_overwrite:
        print(f"Skipping existing file: {final_file}")
        continue

    if partial_file.exists():
        partial_file.unlink()

    print(f"\n{model} / historical: {len(tasks)} member(s)")

    try:
        with tempfile.TemporaryDirectory(
            prefix="eth_hist_tas_"
        ) as temporary_directory:
            temporary_directory = Path(temporary_directory)

            for member_index, task in enumerate(tasks):
                member = task["member"]
                temporary_member_file = (
                    temporary_directory / f"{member}_tas.nc"
                )

                print(f"    tas: {member}")

                create_eth_member_tas(
                    task,
                    temporary_member_file,
                )
                validate_eth_member_file(
                    temporary_member_file,
                    "tas",
                )
                copy_hist_member_to_combined(
                    temporary_member_file,
                    partial_file,
                    "tas",
                    member_index,
                    member,
                    model,
                )

                temporary_member_file.unlink()

        partial_file.replace(final_file)

        print(f"  Saved: {final_file}")
        subprocess.run(
            ["du", "-h", str(final_file)],
            check=True,
        )

    except Exception:
        print(
            f"  FAILED: incomplete output retained at "
            f"{partial_file}"
        )
        raise


ACCESS-CM2 / historical: 3 member(s)
    tas: r1i1p1f1
    tas: r2i1p1f1
    tas: r3i1p1f1
  Saved: data/HIST/ACCESS-CM2/tas_Amon_ACCESS-CM2_historical_combined_187001-201412_g025.nc
79M	data/HIST/ACCESS-CM2/tas_Amon_ACCESS-CM2_historical_combined_187001-201412_g025.nc

ACCESS-ESM1-5 / historical: 30 member(s)
    tas: r1i1p1f1
    tas: r2i1p1f1
    tas: r3i1p1f1
    tas: r4i1p1f1
    tas: r5i1p1f1
    tas: r6i1p1f1
    tas: r7i1p1f1
    tas: r8i1p1f1
    tas: r9i1p1f1
    tas: r10i1p1f1
    tas: r11i1p1f1
    tas: r12i1p1f1
    tas: r13i1p1f1
    tas: r14i1p1f1
    tas: r15i1p1f1
    tas: r16i1p1f1
    tas: r17i1p1f1
    tas: r18i1p1f1
    tas: r19i1p1f1
    tas: r20i1p1f1
    tas: r21i1p1f1
    tas: r22i1p1f1
    tas: r23i1p1f1
    tas: r24i1p1f1
    tas: r25i1p1f1
    tas: r26i1p1f1
    tas: r27i1p1f1
    tas: r28i1p1f1
    tas: r29i1p1f1
    tas: r30i1p1f1
  Saved: data/HIST/ACCESS-ESM1-5/tas_Amon_ACCESS-ESM1-5_historical_combined_187001-201412_g025.nc
1.2G	data/HIST/ACCESS-ESM1-5

In [7]:
# Cell 5 — Verify every HIST N/tas pair
hist_verification_failures = []

for model, tasks in sorted(eth_hist_tasks_by_model.items()):
    model_directory = hist_output_root / model

    n_file = (
        model_directory
        / (
            f"N_Amon_{model}_historical_combined_"
            f"{hist_output_period}_g025.nc"
        )
    )
    tas_file = (
        model_directory
        / (
            f"tas_Amon_{model}_historical_combined_"
            f"{hist_output_period}_g025.nc"
        )
    )

    issues = []

    if not n_file.exists():
        issues.append("N output is missing")

    if not tas_file.exists():
        issues.append("tas output is missing")

    if not issues:
        with (
            Dataset(n_file, "r") as n_dataset,
            Dataset(tas_file, "r") as tas_dataset,
        ):
            n_members = [
                str(value)
                for value in n_dataset.variables["member"][:]
            ]
            tas_members = [
                str(value)
                for value in tas_dataset.variables["member"][:]
            ]

            expected_members = [
                task["member"]
                for task in tasks
            ]

            if n_members != expected_members:
                issues.append(
                    "N member labels or ordering differ from inventory"
                )

            if tas_members != expected_members:
                issues.append(
                    "tas member labels or ordering differ from inventory"
                )

            if n_members != tas_members:
                issues.append(
                    "N and tas member labels differ"
                )

            if (
                n_dataset.variables["N"].shape
                != tas_dataset.variables["tas"].shape
            ):
                issues.append(
                    f"N/tas shapes differ: "
                    f"{n_dataset.variables['N'].shape} versus "
                    f"{tas_dataset.variables['tas'].shape}"
                )

            if (
                n_dataset.variables["N"].dimensions
                != ("member", "time", "lat", "lon")
            ):
                issues.append(
                    "N dimensions are not member,time,lat,lon"
                )

            if (
                tas_dataset.variables["tas"].dimensions
                != ("member", "time", "lat", "lon")
            ):
                issues.append(
                    "tas dimensions are not member,time,lat,lon"
                )

            with Dataset(hist_g025_reference, "r") as reference_dataset:
                for coordinate in ("lat", "lon"):
                    if not np.allclose(
                        n_dataset.variables[coordinate][:],
                        tas_dataset.variables[coordinate][:],
                        equal_nan=True,
                    ):
                        issues.append(
                            f"N and tas {coordinate} coordinates differ"
                        )

                    if not np.allclose(
                        n_dataset.variables[coordinate][:],
                        reference_dataset.variables[coordinate][:],
                        equal_nan=True,
                    ):
                        issues.append(
                            f"{coordinate} differs from the AMIP g025 grid"
                        )

            if (
                hist_month_sequence(n_dataset)
                != expected_hist_months
            ):
                issues.append(
                    "N time coverage is not exactly 1870-01–2014-12"
                )

            if (
                hist_month_sequence(tas_dataset)
                != expected_hist_months
            ):
                issues.append(
                    "tas time coverage is not exactly 1870-01–2014-12"
                )

    if issues:
        hist_verification_failures.append(
            (model, issues)
        )
        print(
            f"PROBLEM  {model:<28}"
            + "; ".join(issues)
        )
    else:
        print(
            f"OK       {model:<28}"
            f"{len(tasks):>3} member(s)"
        )

if hist_verification_failures:
    raise RuntimeError(
        f"{len(hist_verification_failures)} HIST verification(s) failed"
    )

print(
    "\nAll HIST N/tas pairs have matching members, "
    "1870–2014 coverage, and the AMIP g025 grid."
)

OK       ACCESS-CM2                    3 member(s)
OK       ACCESS-ESM1-5                30 member(s)
OK       AWI-CM-1-1-MR                 5 member(s)
OK       AWI-ESM-1-1-LR                1 member(s)
OK       BCC-CSM2-MR                   3 member(s)
OK       BCC-ESM1                      3 member(s)
OK       CAMS-CSM1-0                   3 member(s)
OK       CAS-ESM2-0                    4 member(s)
OK       CESM2                        11 member(s)
OK       CESM2-FV2                     3 member(s)
OK       CESM2-WACCM                   3 member(s)
OK       CESM2-WACCM-FV2               3 member(s)
OK       CIESM                         3 member(s)
OK       CMCC-CM2-HR4                  1 member(s)
OK       CMCC-CM2-SR5                  1 member(s)
OK       CMCC-ESM2                     1 member(s)
OK       CNRM-CM6-1                   29 member(s)
OK       CNRM-CM6-1-HR                 1 member(s)
OK       CNRM-ESM2-1                  10 member(s)
OK       CanESM5               